# Libaries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

from src.benchmark import ElasticityConfig
from src.dominick import DominickDataLoader
from src.utils import TemporalSplitter, BlockBootstrapSampler

In [2]:
TRAIN_FRAC = 0.8
N_FOLDS = 5
N_BOOTSTRAP = 20

SELECTED_UPCS = [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]

# Loader

In [3]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()

df = df[df["upc_code"].isin(SELECTED_UPCS)].copy()

print(f"Dataset shape: {df.shape}")
print(f"N semanas: {df['week_id'].nunique()}")
print(f"N productos seleccionados: {df['upc_code'].nunique()}")
print(sorted(df["upc_code"].unique()))

Dataset shape: (73209, 30)
N semanas: 302
N productos seleccionados: 5
[1820000784, 3410010505, 3410017306, 7289000011, 8248812345]


# Config + Pipeline

In [4]:
config = ElasticityConfig(
    csv_path="elasticity_dataset.csv",
    cross_price_cols=[],
)

splitter = TemporalSplitter(week_col="week_id")
control_cols = config.numeric_control_cols.copy()

# Pairs

In [5]:
base_cols = [
    "store_code",
    "week_id",
    "upc_code",
    "log_liters_sold",
    "log_price_per_liter",
]

pair_base = df[base_cols + control_cols].copy()

left = pair_base.rename(columns={
    "upc_code": "upc_i",
    "log_liters_sold": "log_v_i",
    "log_price_per_liter": "log_p_i",
})

right = pair_base.rename(columns={
    "upc_code": "upc_j",
    "log_price_per_liter": "log_p_j",
})

pair_df = left.merge(
    right[["store_code", "week_id", "upc_j", "log_p_j"]],
    on=["store_code", "week_id"],
    how="inner",
)

pair_df = pair_df[pair_df["upc_i"] != pair_df["upc_j"]].copy()

pair_df["pair_id"] = pair_df.apply(
    lambda r: "__".join(map(str, sorted([r["upc_i"], r["upc_j"]]))),
    axis=1,
)

print(f"Pair dataset shape: {pair_df.shape}")
display(pair_df.head(10))

Pair dataset shape: (225004, 32)


,store_code,week_id,upc_i,log_v_i,log_p_i,on_promo,week_rank,sin_52,cos_52,sin_26,...,miss_lag_1,miss_lag_2,miss_lag_4,miss_roll_4,miss_roll_8,miss_roll_13,promo_intensity_store_week,upc_j,log_p_j,pair_id
1,5,166,3410017306,3.240697,0.341957,0.0,76,2.393157e-01,-0.970942,-4.647232e-01,...,1,1,1,1,1,1,0.0,7289000011,1.114486,3410017306.0__7289000011.0
2,5,166,7289000011,0.755790,1.114486,0.0,76,2.393157e-01,-0.970942,-4.647232e-01,...,1,1,1,1,1,1,0.0,3410017306,0.341957,3410017306.0__7289000011.0
6,5,168,3410017306,4.850135,0.254875,1.0,78,3.673940e-16,-1.000000,-7.347881e-16,...,0,0,1,0,0,0,0.2,7289000011,1.114486,3410017306.0__7289000011.0
7,5,168,7289000011,0.755790,1.114486,0.0,78,3.673940e-16,-1.000000,-7.347881e-16,...,1,0,1,0,0,0,0.2,3410017306,0.254875,3410017306.0__7289000011.0
12,5,171,3410017306,5.086524,0.254875,1.0,81,-3.546049e-01,-0.935016,6.631227e-01,...,0,0,0,0,0,0,0.6,7289000011,1.114486,3410017306.0__7289000011.0
13,5,171,7289000011,1.448938,1.114486,0.0,81,-3.546049e-01,-0.935016,6.631227e-01,...,1,1,1,0,0,0,0.6,3410017306,0.254875,3410017306.0__7289000011.0
19,5,175,3410017306,3.240697,0.341957,0.0,85,-7.485107e-01,-0.663123,9.927089e-01,...,0,0,0,0,0,0,0.2,7289000011,1.114486,3410017306.0__7289000011.0
20,5,175,7289000011,1.854403,1.114486,0.0,85,-7.485107e-01,-0.663123,9.927089e-01,...,1,1,0,0,0,0,0.2,3410017306,0.341957,3410017306.0__7289000011.0
25,5,178,3410017306,3.528379,0.341957,0.0,88,-9.350162e-01,-0.354605,6.631227e-01,...,0,0,0,0,0,0,0.0,7289000011,1.114486,3410017306.0__7289000011.0
26,5,178,7289000011,1.448938,1.114486,0.0,88,-9.350162e-01,-0.354605,6.631227e-01,...,1,1,1,0,0,0,0.0,3410017306,0.341957,3410017306.0__7289000011.0


In [6]:
def fit_pairwise_elasticities(train_pair_df, val_pair_df, control_cols, min_obs=30, robust_cov_type="HC1"):
    results = []

    train_groups = {
        k: g.copy()
        for k, g in train_pair_df.groupby(["store_code", "pair_id", "upc_i", "upc_j"])
    }
    val_groups = {
        k: g.copy()
        for k, g in val_pair_df.groupby(["store_code", "pair_id", "upc_i", "upc_j"])
    }

    common_keys = sorted(set(train_groups.keys()) & set(val_groups.keys()))

    for key in common_keys:
        store_code, pair_id, upc_i, upc_j = key

        g_train = train_groups[key].copy()
        g_val = val_groups[key].copy()

        needed_cols = ["log_v_i", "log_p_i", "log_p_j"] + control_cols
        g_train = g_train[needed_cols].dropna()
        g_val = g_val[needed_cols].dropna()

        if len(g_train) < min_obs:
            continue

        if len(g_val) == 0:
            continue

        if g_train["log_p_i"].nunique() < 2:
            continue

        if g_train["log_p_j"].nunique() < 2:
            continue

        formula = "log_v_i ~ log_p_i + log_p_j"
        if len(control_cols) > 0:
            formula += " + " + " + ".join(control_cols)

        try:
            fit = smf.ols(formula=formula, data=g_train).fit(cov_type=robust_cov_type)

            pred_val = fit.predict(g_val)

            mae_val = np.mean(np.abs(g_val["log_v_i"] - pred_val))
            rmse_val = np.sqrt(np.mean((g_val["log_v_i"] - pred_val) ** 2))

            ss_res = np.sum((g_val["log_v_i"] - pred_val) ** 2)
            ss_tot = np.sum((g_val["log_v_i"] - g_val["log_v_i"].mean()) ** 2)
            r2_val = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot

            ci = fit.conf_int()

            results.append({
                "store_code": store_code,
                "pair_id": pair_id,
                "upc_i": upc_i,
                "upc_j": upc_j,
                "status": "ok",
                "n_train": len(g_train),
                "n_val": len(g_val),

                "own_elasticity": fit.params.get("log_p_i", np.nan),
                "own_elasticity_ci_low": ci.loc["log_p_i", 0] if "log_p_i" in ci.index else np.nan,
                "own_elasticity_ci_high": ci.loc["log_p_i", 1] if "log_p_i" in ci.index else np.nan,
                "own_elasticity_p_value": fit.pvalues.get("log_p_i", np.nan),

                "cross_elasticity": fit.params.get("log_p_j", np.nan),
                "cross_elasticity_ci_low": ci.loc["log_p_j", 0] if "log_p_j" in ci.index else np.nan,
                "cross_elasticity_ci_high": ci.loc["log_p_j", 1] if "log_p_j" in ci.index else np.nan,
                "cross_elasticity_p_value": fit.pvalues.get("log_p_j", np.nan),

                "mae_val": mae_val,
                "rmse_val": rmse_val,
                "r2_val": r2_val,
            })

        except Exception as exc:
            results.append({
                "store_code": store_code,
                "pair_id": pair_id,
                "upc_i": upc_i,
                "upc_j": upc_j,
                "status": "error",
                "error_message": str(exc),
            })

    return pd.DataFrame(results)

In [7]:
def symmetrize_cross_elasticities(df_cross):
    if df_cross.empty:
        return pd.DataFrame(columns=[
            "store_code", "pair_id", "upc_a", "upc_b",
            "cross_elasticity_sym", "n_directions",
            "avg_p_value", "mae_val_mean", "rmse_val_mean", "r2_val_mean"
        ])

    tmp = df_cross[df_cross["status"] == "ok"].copy()

    tmp["upc_a"] = tmp[["upc_i", "upc_j"]].min(axis=1)
    tmp["upc_b"] = tmp[["upc_i", "upc_j"]].max(axis=1)

    sym_df = (
        tmp
        .groupby(["store_code", "pair_id", "upc_a", "upc_b"], as_index=False)
        .agg(
            cross_elasticity_sym=("cross_elasticity", "mean"),
            n_directions=("cross_elasticity", "count"),
            avg_p_value=("cross_elasticity_p_value", "mean"),
            mae_val_mean=("mae_val", "mean"),
            rmse_val_mean=("rmse_val", "mean"),
            r2_val_mean=("r2_val", "mean"),
        )
    )

    return sym_df

# Evaluation K-fold

In [8]:
fold_results = []

for fold_idx, (train_fold, val_fold) in enumerate(splitter.expanding_splits(pair_df, N_FOLDS)):
    results = fit_pairwise_elasticities(
        train_pair_df=train_fold,
        val_pair_df=val_fold,
        control_cols=control_cols,
        min_obs=config.min_obs,
        robust_cov_type=config.robust_cov_type,
    )
    results["fold"] = fold_idx
    fold_results.append(results)

all_folds = pd.concat(fold_results, ignore_index=True) if len(fold_results) > 0 else pd.DataFrame()
ok_folds = all_folds[all_folds["status"] == "ok"].copy()

benchmark_generalization_folds = ok_folds[[
    "store_code",
    "pair_id",
    "upc_i",
    "upc_j",
    "fold",
    "own_elasticity",
    "own_elasticity_ci_low",
    "own_elasticity_ci_high",
    "own_elasticity_p_value",
    "cross_elasticity",
    "cross_elasticity_ci_low",
    "cross_elasticity_ci_high",
    "cross_elasticity_p_value",
    "mae_val",
    "rmse_val",
    "r2_val",
]].copy()

benchmark_cross_generalization_folds = symmetrize_cross_elasticities(ok_folds)

if not benchmark_cross_generalization_folds.empty:
    benchmark_cross_generalization_folds = benchmark_cross_generalization_folds.merge(
        ok_folds[["store_code", "pair_id", "fold"]].drop_duplicates(),
        on=["store_code", "pair_id"],
        how="left",
    )

print(f"K-fold raw: {len(benchmark_generalization_folds)} filas")
print(f"K-fold symmetric cross: {len(benchmark_cross_generalization_folds)} filas")

K-fold raw: 3944 filas
K-fold symmetric cross: 1972 filas


# Evaluation Bootstrap

In [9]:
train_df, val_df = splitter.single_split(pair_df, train_frac=TRAIN_FRAC)
train_weeks = sorted(train_df["week_id"].unique())

sampler = BlockBootstrapSampler(
    week_col="week_id",
    block_size=4,
    rng=np.random.default_rng(42),
)

bootstrap_results = []

for b in range(N_BOOTSTRAP):
    train_bs = sampler.sample(train_df, train_weeks)

    res = fit_pairwise_elasticities(
        train_pair_df=train_bs,
        val_pair_df=val_df,
        control_cols=control_cols,
        min_obs=config.min_obs,
        robust_cov_type=config.robust_cov_type,
    )
    res["bootstrap_run"] = b
    bootstrap_results.append(res)

all_bootstrap = pd.concat(bootstrap_results, ignore_index=True) if len(bootstrap_results) > 0 else pd.DataFrame()
ok_bs = all_bootstrap[all_bootstrap["status"] == "ok"].copy()

benchmark_elasticity_bootstrap_raw = ok_bs[[
    "store_code",
    "pair_id",
    "upc_i",
    "upc_j",
    "bootstrap_run",
    "own_elasticity",
    "cross_elasticity",
    "mae_val",
    "rmse_val",
    "r2_val",
]].copy()

print(f"Bootstrap raw: {len(benchmark_elasticity_bootstrap_raw)} filas")

Bootstrap raw: 10880 filas


In [10]:
def q025(x):
    return np.percentile(x, 2.5)

def q975(x):
    return np.percentile(x, 97.5)

benchmark_elasticities_bootstrap_summary = (
    ok_bs
    .groupby(["store_code", "pair_id", "upc_i", "upc_j"])
    .agg(
        own_elasticity_mean=("own_elasticity", "mean"),
        own_elasticity_std=("own_elasticity", "std"),
        own_elasticity_ci_low=("own_elasticity", q025),
        own_elasticity_ci_high=("own_elasticity", q975),

        cross_elasticity_mean=("cross_elasticity", "mean"),
        cross_elasticity_std=("cross_elasticity", "std"),
        cross_elasticity_ci_low=("cross_elasticity", q025),
        cross_elasticity_ci_high=("cross_elasticity", q975),

        mae_val_mean=("mae_val", "mean"),
        mae_val_std=("mae_val", "std"),
        rmse_val_mean=("rmse_val", "mean"),
        rmse_val_std=("rmse_val", "std"),
        r2_val_mean=("r2_val", "mean"),
        r2_val_std=("r2_val", "std"),
    )
    .reset_index()
)

print(f"Bootstrap summary: {len(benchmark_elasticities_bootstrap_summary)} filas")
display(benchmark_elasticities_bootstrap_summary.head(10))

Bootstrap summary: 544 filas


,store_code,pair_id,upc_i,upc_j,own_elasticity_mean,own_elasticity_std,own_elasticity_ci_low,own_elasticity_ci_high,cross_elasticity_mean,cross_elasticity_std,cross_elasticity_ci_low,cross_elasticity_ci_high,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std
0,8,1820000784.0__3410010505.0,1820000784,3410010505,-3.964395,0.680193,-4.921938,-2.639270,-0.376021,0.325806,-0.961702,0.087784,0.524610,0.032588,0.707262,0.036821,-0.038334,0.108684
1,8,1820000784.0__3410010505.0,3410010505,1820000784,-3.426151,0.505120,-4.243194,-2.515982,0.746976,0.546606,-0.273952,1.524663,0.629345,0.067967,0.768118,0.071242,-0.892317,0.352611
2,8,1820000784.0__7289000011.0,1820000784,7289000011,-3.618175,0.925191,-4.972160,-1.997445,-2.743262,1.146252,-4.278193,-0.438170,0.598903,0.040358,0.768747,0.049698,-0.228423,0.158863
3,8,1820000784.0__7289000011.0,7289000011,1820000784,-0.293997,1.954698,-4.048812,2.727934,-0.762013,0.507922,-1.587889,0.110380,0.656350,0.088843,0.823043,0.096157,-0.456103,0.357755
4,8,3410010505.0__7289000011.0,3410010505,7289000011,-2.931868,0.610548,-3.768094,-1.912406,-0.906398,1.003366,-3.277282,0.380037,0.607058,0.071369,0.735456,0.081244,-0.740695,0.400911
5,8,3410010505.0__7289000011.0,7289000011,3410010505,-1.045188,2.319066,-5.245457,2.509903,-0.821285,0.626210,-1.492621,0.384377,0.671893,0.079762,0.836068,0.085923,-0.498203,0.322734
6,9,1820000784.0__3410010505.0,1820000784,3410010505,-4.867760,0.803384,-6.638667,-3.635060,0.555180,0.530717,-0.543625,1.266655,1.416517,0.162801,1.626091,0.178238,-4.965926,1.283685
7,9,1820000784.0__3410010505.0,3410010505,1820000784,-3.035237,0.404587,-3.604666,-2.331240,0.698938,0.509137,-0.146390,1.534889,0.415358,0.041936,0.503438,0.050358,-0.289171,0.263211
8,9,1820000784.0__7289000011.0,1820000784,7289000011,-4.403238,0.818527,-5.868370,-3.142305,-0.504029,1.005135,-2.169225,1.129105,1.313337,0.187167,1.515422,0.204940,-4.212033,1.401959
9,9,1820000784.0__7289000011.0,7289000011,1820000784,-2.832014,2.380764,-6.815087,0.941460,1.035438,0.429916,0.335254,1.756877,0.437672,0.056006,0.560536,0.051642,0.255687,0.135479


In [11]:
cross_bootstrap_sym = []

for b in sorted(ok_bs["bootstrap_run"].dropna().unique()):
    tmp_b = ok_bs[ok_bs["bootstrap_run"] == b].copy()
    sym_b = symmetrize_cross_elasticities(tmp_b)
    sym_b["bootstrap_run"] = b
    cross_bootstrap_sym.append(sym_b)

benchmark_cross_bootstrap_sym_raw = (
    pd.concat(cross_bootstrap_sym, ignore_index=True)
    if len(cross_bootstrap_sym) > 0 else pd.DataFrame()
)

benchmark_cross_bootstrap_summary = (
    benchmark_cross_bootstrap_sym_raw
    .groupby(["store_code", "pair_id", "upc_a", "upc_b"])
    .agg(
        cross_elasticity_sym_mean=("cross_elasticity_sym", "mean"),
        cross_elasticity_sym_std=("cross_elasticity_sym", "std"),
        cross_elasticity_sym_ci_low=("cross_elasticity_sym", q025),
        cross_elasticity_sym_ci_high=("cross_elasticity_sym", q975),
        mae_val_mean=("mae_val_mean", "mean"),
        mae_val_std=("mae_val_mean", "std"),
        rmse_val_mean=("rmse_val_mean", "mean"),
        rmse_val_std=("rmse_val_mean", "std"),
        r2_val_mean=("r2_val_mean", "mean"),
        r2_val_std=("r2_val_mean", "std"),
    )
    .reset_index()
)

print(f"Bootstrap symmetric cross raw: {len(benchmark_cross_bootstrap_sym_raw)} filas")
print(f"Bootstrap symmetric cross summary: {len(benchmark_cross_bootstrap_summary)} filas")

Bootstrap symmetric cross raw: 5440 filas
Bootstrap symmetric cross summary: 272 filas


# Results

In [12]:
display(benchmark_generalization_folds.head(10))
display(benchmark_generalization_folds[[
    "own_elasticity", "cross_elasticity", "mae_val", "rmse_val", "r2_val"
]].describe())

display(benchmark_elasticity_bootstrap_raw.head(10))
display(benchmark_elasticities_bootstrap_summary.head(10))
display(benchmark_cross_generalization_folds.head(10))
display(benchmark_cross_bootstrap_sym_raw.head(10))
display(benchmark_cross_bootstrap_summary.head(10))

display(all_folds["status"].value_counts(dropna=False) if not all_folds.empty else "No kfold results")
display(all_bootstrap["status"].value_counts(dropna=False) if not all_bootstrap.empty else "No bootstrap results")

,store_code,pair_id,upc_i,upc_j,fold,own_elasticity,own_elasticity_ci_low,own_elasticity_ci_high,own_elasticity_p_value,cross_elasticity,cross_elasticity_ci_low,cross_elasticity_ci_high,cross_elasticity_p_value,mae_val,rmse_val,r2_val
0,5,3410017306.0__7289000011.0,3410017306,7289000011,0,4.106132,-6.420383,14.632647,4.445495e-01,-7.627939,-24.380248,9.124370,0.372155,1.291065,1.674757,-6.916222
1,5,3410017306.0__7289000011.0,7289000011,3410017306,0,8.222317,-23.156471,39.601104,6.075473e-01,-0.493523,-8.192901,7.205856,0.900023,1.039971,1.293768,-3.872342
2,8,1820000784.0__3410010505.0,1820000784,3410010505,0,-4.142325,-5.972335,-2.312314,9.144133e-06,-1.300767,-2.715340,0.113806,0.071501,0.667487,0.854567,-0.111614
3,8,1820000784.0__3410010505.0,3410010505,1820000784,0,-5.176662,-7.208852,-3.144471,5.954817e-07,-0.322594,-1.463468,0.818281,0.579442,0.782885,0.939221,-1.195786
4,8,1820000784.0__3410017306.0,1820000784,3410017306,0,-4.776690,-6.536305,-3.017075,1.034471e-07,0.811288,-1.669465,3.292042,0.521540,0.594284,0.743710,0.158083
5,8,1820000784.0__3410017306.0,3410017306,1820000784,0,-6.256416,-8.409012,-4.103820,1.222628e-08,0.137269,-1.081549,1.356087,0.825295,0.598531,0.779146,-1.384955
6,8,1820000784.0__7289000011.0,1820000784,7289000011,0,-3.493284,-5.736007,-1.250561,2.266749e-03,-0.999529,-7.877796,5.878738,0.775785,0.595662,0.770598,0.123116
7,8,1820000784.0__7289000011.0,7289000011,1820000784,0,-1.897329,-9.597517,5.802858,6.291412e-01,-0.708827,-2.130977,0.713323,0.328627,0.523370,0.639952,0.022538
8,8,3410010505.0__3410017306.0,3410010505,3410017306,0,-5.750082,-7.820377,-3.679786,5.220171e-08,0.684322,-1.352670,2.721314,0.510253,0.904389,1.067756,-1.837910
9,8,3410010505.0__3410017306.0,3410017306,3410010505,0,-6.172083,-8.429384,-3.914783,8.364557e-08,-0.789207,-1.930209,0.351794,0.175205,0.482902,0.627527,-0.547061


,own_elasticity,cross_elasticity,mae_val,rmse_val,r2_val
count,3944.000000,3944.000000,3944.000000,3944.000000,3944.000000
mean,-3.760403,-0.197875,0.524869,0.652909,-0.559355
std,2.438620,1.447692,0.247630,0.278161,5.313071
min,-13.526105,-15.425894,0.121328,0.151870,-211.250510
25%,-5.303098,-0.701600,0.396473,0.500785,-0.482744
50%,-3.814866,-0.229523,0.487237,0.616125,-0.027512
75%,-2.332335,0.258226,0.587101,0.735471,0.295084
max,20.926926,15.496618,4.921428,5.314794,0.895395


,store_code,pair_id,upc_i,upc_j,bootstrap_run,own_elasticity,cross_elasticity,mae_val,rmse_val,r2_val
0,8,1820000784.0__3410010505.0,1820000784,3410010505,0,-4.477610,-0.236069,0.517469,0.709651,-0.042675
1,8,1820000784.0__3410010505.0,3410010505,1820000784,0,-2.938823,1.019129,0.506036,0.636882,-0.290389
2,8,1820000784.0__7289000011.0,1820000784,7289000011,0,-3.461365,-4.309116,0.617126,0.786185,-0.279704
3,8,1820000784.0__7289000011.0,7289000011,1820000784,0,1.227771,-0.604761,0.726897,0.920577,-0.798340
4,8,3410010505.0__7289000011.0,3410010505,7289000011,0,-3.142483,-2.403934,0.558803,0.669625,-0.426483
5,8,3410010505.0__7289000011.0,7289000011,3410010505,0,1.190070,0.238843,0.712708,0.924825,-0.814974
6,9,1820000784.0__3410010505.0,1820000784,3410010505,0,-4.701974,1.077400,1.590291,1.835110,-6.512478
7,9,1820000784.0__3410010505.0,3410010505,1820000784,0,-2.540931,0.947026,0.433861,0.516521,-0.344266
8,9,1820000784.0__7289000011.0,1820000784,7289000011,0,-4.048070,-0.658116,1.543470,1.791560,-6.160141
9,9,1820000784.0__7289000011.0,7289000011,1820000784,0,-1.191491,1.172296,0.511710,0.621455,0.092429


,store_code,pair_id,upc_i,upc_j,own_elasticity_mean,own_elasticity_std,own_elasticity_ci_low,own_elasticity_ci_high,cross_elasticity_mean,cross_elasticity_std,cross_elasticity_ci_low,cross_elasticity_ci_high,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std
0,8,1820000784.0__3410010505.0,1820000784,3410010505,-3.964395,0.680193,-4.921938,-2.639270,-0.376021,0.325806,-0.961702,0.087784,0.524610,0.032588,0.707262,0.036821,-0.038334,0.108684
1,8,1820000784.0__3410010505.0,3410010505,1820000784,-3.426151,0.505120,-4.243194,-2.515982,0.746976,0.546606,-0.273952,1.524663,0.629345,0.067967,0.768118,0.071242,-0.892317,0.352611
2,8,1820000784.0__7289000011.0,1820000784,7289000011,-3.618175,0.925191,-4.972160,-1.997445,-2.743262,1.146252,-4.278193,-0.438170,0.598903,0.040358,0.768747,0.049698,-0.228423,0.158863
3,8,1820000784.0__7289000011.0,7289000011,1820000784,-0.293997,1.954698,-4.048812,2.727934,-0.762013,0.507922,-1.587889,0.110380,0.656350,0.088843,0.823043,0.096157,-0.456103,0.357755
4,8,3410010505.0__7289000011.0,3410010505,7289000011,-2.931868,0.610548,-3.768094,-1.912406,-0.906398,1.003366,-3.277282,0.380037,0.607058,0.071369,0.735456,0.081244,-0.740695,0.400911
5,8,3410010505.0__7289000011.0,7289000011,3410010505,-1.045188,2.319066,-5.245457,2.509903,-0.821285,0.626210,-1.492621,0.384377,0.671893,0.079762,0.836068,0.085923,-0.498203,0.322734
6,9,1820000784.0__3410010505.0,1820000784,3410010505,-4.867760,0.803384,-6.638667,-3.635060,0.555180,0.530717,-0.543625,1.266655,1.416517,0.162801,1.626091,0.178238,-4.965926,1.283685
7,9,1820000784.0__3410010505.0,3410010505,1820000784,-3.035237,0.404587,-3.604666,-2.331240,0.698938,0.509137,-0.146390,1.534889,0.415358,0.041936,0.503438,0.050358,-0.289171,0.263211
8,9,1820000784.0__7289000011.0,1820000784,7289000011,-4.403238,0.818527,-5.868370,-3.142305,-0.504029,1.005135,-2.169225,1.129105,1.313337,0.187167,1.515422,0.204940,-4.212033,1.401959
9,9,1820000784.0__7289000011.0,7289000011,1820000784,-2.832014,2.380764,-6.815087,0.941460,1.035438,0.429916,0.335254,1.756877,0.437672,0.056006,0.560536,0.051642,0.255687,0.135479


,store_code,pair_id,upc_a,upc_b,cross_elasticity_sym,n_directions,avg_p_value,mae_val_mean,rmse_val_mean,r2_val_mean,fold
0,5,3410017306.0__7289000011.0,3410017306,7289000011,-1.383122,6,0.637684,0.798330,0.987630,-2.433077,0
1,5,3410017306.0__7289000011.0,3410017306,7289000011,-1.383122,6,0.637684,0.798330,0.987630,-2.433077,1
2,5,3410017306.0__7289000011.0,3410017306,7289000011,-1.383122,6,0.637684,0.798330,0.987630,-2.433077,2
3,8,1820000784.0__3410010505.0,1820000784,3410010505,-0.038279,10,0.445773,0.577227,0.720697,-0.319545,0
4,8,1820000784.0__3410010505.0,1820000784,3410010505,-0.038279,10,0.445773,0.577227,0.720697,-0.319545,1
5,8,1820000784.0__3410010505.0,1820000784,3410010505,-0.038279,10,0.445773,0.577227,0.720697,-0.319545,2
6,8,1820000784.0__3410010505.0,1820000784,3410010505,-0.038279,10,0.445773,0.577227,0.720697,-0.319545,3
7,8,1820000784.0__3410010505.0,1820000784,3410010505,-0.038279,10,0.445773,0.577227,0.720697,-0.319545,4
8,8,1820000784.0__3410017306.0,1820000784,3410017306,0.043701,6,0.684525,0.575787,0.695594,-0.465188,0
9,8,1820000784.0__3410017306.0,1820000784,3410017306,0.043701,6,0.684525,0.575787,0.695594,-0.465188,1


,store_code,pair_id,upc_a,upc_b,cross_elasticity_sym,n_directions,avg_p_value,mae_val_mean,rmse_val_mean,r2_val_mean,bootstrap_run
0,8,1820000784.0__3410010505.0,1820000784,3410010505,0.391530,2,0.318706,0.511753,0.673266,-0.166532,0
1,8,1820000784.0__7289000011.0,1820000784,7289000011,-2.456938,2,0.160046,0.672011,0.853381,-0.539022,0
2,8,3410010505.0__7289000011.0,3410010505,7289000011,-1.082546,2,0.335128,0.635755,0.797225,-0.620728,0
3,9,1820000784.0__3410010505.0,1820000784,3410010505,1.012213,2,0.017196,1.012076,1.175815,-3.428372,0
4,9,1820000784.0__7289000011.0,1820000784,7289000011,0.257090,2,0.252991,1.027590,1.206507,-3.033856,0
5,9,1820000784.0__8248812345.0,1820000784,8248812345,-0.093733,2,0.622094,1.062812,1.221458,-2.858910,0
6,9,3410010505.0__7289000011.0,3410010505,7289000011,0.001655,2,0.275536,0.537795,0.631280,-0.188413,0
7,9,3410010505.0__8248812345.0,3410010505,8248812345,-0.482252,2,0.177616,0.489495,0.581941,0.178046,0
8,9,7289000011.0__8248812345.0,7289000011,8248812345,1.610373,2,0.355745,0.605181,0.724985,0.071458,0
9,12,1820000784.0__3410010505.0,1820000784,3410010505,0.545735,2,0.421537,0.500124,0.604087,-0.041489,0


,store_code,pair_id,upc_a,upc_b,cross_elasticity_sym_mean,cross_elasticity_sym_std,cross_elasticity_sym_ci_low,cross_elasticity_sym_ci_high,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std
0,8,1820000784.0__3410010505.0,1820000784,3410010505,0.185477,0.314908,-0.417184,0.630393,0.576978,0.032577,0.737690,0.032506,-0.465325,0.160917
1,8,1820000784.0__7289000011.0,1820000784,7289000011,-1.752637,0.672256,-2.521215,-0.452347,0.627626,0.047909,0.795895,0.052112,-0.342263,0.187117
2,8,3410010505.0__7289000011.0,3410010505,7289000011,-0.863841,0.373746,-1.591320,-0.328556,0.639475,0.059672,0.785762,0.065243,-0.619449,0.285241
3,9,1820000784.0__3410010505.0,1820000784,3410010505,0.627059,0.360136,-0.115955,1.154257,0.915937,0.090558,1.064765,0.100390,-2.627548,0.693560
4,9,1820000784.0__7289000011.0,1820000784,7289000011,0.265705,0.509478,-0.661215,1.125004,0.875505,0.095765,1.037979,0.103781,-1.978173,0.700654
5,9,1820000784.0__8248812345.0,1820000784,8248812345,-0.285995,0.237808,-0.799753,-0.020547,1.003154,0.076199,1.157004,0.081413,-2.245340,0.602574
6,9,3410010505.0__7289000011.0,3410010505,7289000011,-0.453369,0.794500,-1.817329,0.826269,0.448792,0.049760,0.551405,0.047753,0.048650,0.160462
7,9,3410010505.0__8248812345.0,3410010505,8248812345,-0.040230,0.288478,-0.484859,0.354703,0.502838,0.032617,0.617718,0.033263,0.078898,0.109749
8,9,7289000011.0__8248812345.0,7289000011,8248812345,0.552724,0.432273,-0.081905,1.411933,0.545152,0.051888,0.667457,0.055818,0.218372,0.134426
9,12,1820000784.0__3410010505.0,1820000784,3410010505,0.465981,0.261436,0.082432,0.876611,0.480509,0.027656,0.605768,0.027459,-0.054139,0.095339


ok    3944
Name: status, dtype: int64

ok    10880
Name: status, dtype: int64

# Save

In [13]:
benchmark_generalization_folds.to_csv("../data/benchmark_generalization_folds.csv", index=False)
benchmark_elasticity_bootstrap_raw.to_csv("../data/benchmark_elasticity_bootstrap_raw.csv", index=False)
benchmark_elasticities_bootstrap_summary.to_csv("../data/benchmark_elasticities_bootstrap_summary.csv", index=False)

benchmark_cross_generalization_folds.to_csv("../data/benchmark_cross_generalization_folds.csv", index=False)
benchmark_cross_bootstrap_sym_raw.to_csv("../data/benchmark_cross_bootstrap_sym_raw.csv", index=False)
benchmark_cross_bootstrap_summary.to_csv("../data/benchmark_cross_bootstrap_summary.csv", index=False)

ok_folds.to_csv("../data/benchmark_kfold_raw.csv", index=False)
ok_bs.to_csv("../data/benchmark_bootstrap_raw.csv", index=False)

print("Guardado:")
print("  - benchmark_generalization_folds.csv")
print("  - benchmark_elasticity_bootstrap_raw.csv")
print("  - benchmark_elasticities_bootstrap_summary.csv")
print("  - benchmark_cross_generalization_folds.csv")
print("  - benchmark_cross_bootstrap_sym_raw.csv")
print("  - benchmark_cross_bootstrap_summary.csv")
print("  - benchmark_kfold_raw.csv")
print("  - benchmark_bootstrap_raw.csv")

Guardado:
  - benchmark_generalization_folds.csv
  - benchmark_elasticity_bootstrap_raw.csv
  - benchmark_elasticities_bootstrap_summary.csv
  - benchmark_cross_generalization_folds.csv
  - benchmark_cross_bootstrap_sym_raw.csv
  - benchmark_cross_bootstrap_summary.csv
  - benchmark_kfold_raw.csv
  - benchmark_bootstrap_raw.csv
